<a href="https://colab.research.google.com/github/heyanugrah/sentimentAnalysisTransformerEncoder/blob/main/fullTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Basic Transformer- Encoder

In [1]:
import torch
import torch.nn as nn
import math

In [2]:
# 1. Define the Vocabulary (Adding '[sep]')
new_vocab = {'<pad>': 0, '<unk>': 1, '<cls>': 2, '[sep]': 3, 'this': 4, 'is': 5, 'a': 6, 'positive': 7, 'sentence': 8, 'learning': 9, 'about': 10, 'transformers': 11, 'fun': 12, 'i': 13, 'hate': 14, 'am': 15, 'so': 16, 'disappointed': 17, 'really': 18, 'enjoyed': 19, 'movie': 20, 'wonderful': 21, 'day': 22, 'for': 23, 'walk': 24, 'product': 25, 'excellent!': 26, 'absolutely': 27, 'dreadful': 28, 'service': 29, 'what': 30, 'terrible': 31, 'experience': 32, 'feeling': 33, 'very': 34, 'happy': 35, 'today': 36, 'boy': 37, 'anugrah': 38}

# 2. Define the Reverse Vocabulary
idx_to_word = {i: word for word, i in new_vocab.items()}

vocab_size = len(new_vocab)

print(f"Vocabulary size is: {vocab_size}")

Vocabulary size is: 39


### Modular Tokenization Functions

To improve clarity and reusability, let's break down the tokenization process into several distinct functions:

1.  `_add_special_tokens_and_segments`: Adds `<cls>` and `[sep]` tokens and initializes segment IDs.
2.  `_convert_to_ids`: Converts token strings to their numerical IDs.
3.  `_pad_ids_and_segments`: Handles padding for both token IDs and segment IDs.
4.  `_create_attention_mask_from_ids`: Generates the attention mask from padded token IDs.
5.  `prepare_transformer_inputs`: An orchestrating function that calls the above helpers to produce all necessary inputs for a Transformer model.

In [3]:
# Helper to add special tokens and initial segment IDs
def _add_special_tokens_and_segments(input_text, vocab):
    tokens = [vocab.get("<cls>", "<cls>")] # Start with <cls>
    segment_ids = [0] # <cls> belongs to segment 0

    # Tokenize input text and add words
    for word in input_text.lower().replace('.', '').split():
        tokens.append(word)
        segment_ids.append(0) # All words in the first sentence belong to segment 0

    tokens.append(vocab.get("[sep]", "[sep]")) # Add [sep]
    segment_ids.append(0) # [sep] belongs to segment 0

    return tokens, segment_ids

# Helper to convert tokens to IDs
def _convert_to_ids(tokens, vocab):
    return [vocab.get(token, vocab["<unk>"]) for token in tokens]

# Helper to pad token IDs and segment IDs
def _pad_ids_and_segments(token_ids, segment_ids, max_seq_len, pad_token_id):
    if len(token_ids) > max_seq_len:
        token_ids = token_ids[:max_seq_len]
        segment_ids = segment_ids[:max_seq_len]
    else:
        padding_length = max_seq_len - len(token_ids)
        token_ids.extend([pad_token_id] * padding_length)
        segment_ids.extend([0] * padding_length) # Padding tokens belong to segment 0

    return token_ids, segment_ids

# Helper to create attention mask
def _create_attention_mask_from_ids(token_ids, pad_token_id):
    mask = [1 if token_id != pad_token_id else 0 for token_id in token_ids]
    return torch.tensor(mask, dtype=torch.long).unsqueeze(0).unsqueeze(1).unsqueeze(2) # [batch_size, 1, 1, seq_len]

In [4]:
def prepare_transformer_inputs(input_text, vocab, max_seq_len):
    pad_token_id = vocab['<pad>']

    # Step 1: Add special tokens and assign initial segment IDs
    tokens, segment_ids = _add_special_tokens_and_segments(input_text, vocab)

    # Step 2: Convert tokens to IDs
    token_ids = _convert_to_ids(tokens, vocab)

    # Step 3: Pad token IDs and segment IDs
    padded_token_ids, padded_segment_ids = _pad_ids_and_segments(
        token_ids, segment_ids, max_seq_len, pad_token_id
    )

    # Convert to PyTorch tensors and add batch dimension
    input_ids = torch.tensor(padded_token_ids, dtype=torch.long).unsqueeze(0)
    token_type_ids = torch.tensor(padded_segment_ids, dtype=torch.long).unsqueeze(0)

    # Step 4: Create attention mask
    attention_mask = _create_attention_mask_from_ids(padded_token_ids, pad_token_id)

    return input_ids, attention_mask, token_type_ids

### Demonstration of the new tokenization pipeline

In [5]:


input_text = "anugrah is learning"
max_seq_len = 10 # maximum token need to be created

# Prepare inputs using the new modular function
input_ids, attention_mask_new, token_type_ids_new = prepare_transformer_inputs(
    input_text, new_vocab, max_seq_len
)

print(f"Input Text: '{input_text}'")
print(f"Input IDs: {input_ids}")
print(f"Attention Mask: {attention_mask_new}")
print(f"Token Type IDs: {token_type_ids_new}")

print(f"Input IDs shape: {input_ids.shape}")
print(f"Attention Mask shape: {attention_mask_new.shape}")
print(f"Token Type IDs shape: {token_type_ids_new.shape}")

Input Text: 'anugrah is learning'
Input IDs: tensor([[ 1, 38,  5,  9,  1,  0,  0,  0,  0,  0]])
Attention Mask: tensor([[[[1, 1, 1, 1, 1, 0, 0, 0, 0, 0]]]])
Token Type IDs: tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
Input IDs shape: torch.Size([1, 10])
Attention Mask shape: torch.Size([1, 1, 1, 10])
Token Type IDs shape: torch.Size([1, 10])


### Create and apply embedding

In [6]:
d_model = 8 #vector dimension for each word

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=d_model
)

embedded_input = embedding(input_ids)
print(f"Shape of embedded input: {embedded_input.shape}")

Shape of embedded input: torch.Size([1, 10, 8])


In [7]:
print(embedded_input)

tensor([[[ 0.0568,  0.7025,  0.5733,  0.7991,  0.1911,  0.0176,  0.4674,
           0.7977],
         [ 0.6001, -1.5990,  0.7913, -0.7446,  0.2965, -1.0727, -0.3515,
          -0.0779],
         [ 1.2538, -1.7399, -0.5808,  0.1176,  2.2297,  0.9909,  0.3007,
          -1.5320],
         [ 0.5762,  1.5890, -0.5824,  0.6182, -0.4549,  1.9881, -0.4141,
           0.1068],
         [ 0.0568,  0.7025,  0.5733,  0.7991,  0.1911,  0.0176,  0.4674,
           0.7977],
         [ 0.2787,  0.8736, -0.5239,  0.4374,  0.5422, -0.9520, -0.4501,
          -0.0703],
         [ 0.2787,  0.8736, -0.5239,  0.4374,  0.5422, -0.9520, -0.4501,
          -0.0703],
         [ 0.2787,  0.8736, -0.5239,  0.4374,  0.5422, -0.9520, -0.4501,
          -0.0703],
         [ 0.2787,  0.8736, -0.5239,  0.4374,  0.5422, -0.9520, -0.4501,
          -0.0703],
         [ 0.2787,  0.8736, -0.5239,  0.4374,  0.5422, -0.9520, -0.4501,
          -0.0703]]], grad_fn=<EmbeddingBackward0>)


In [8]:
print(f"Input Text: '{input_text}'")
print(f"Input IDs: {input_ids.squeeze(0)}")

print("\nEncodings for each token in 'anugrah is learning':")
# Iterate through the input_ids and corresponding embedded vectors
for i, token_id in enumerate(input_ids.squeeze(0)):
    word = idx_to_word.get(token_id.item(), '<unknown>')
    embedding_vector = embedded_input.squeeze(0)[i]
    print(f"Token: '{word}' (ID: {token_id.item()}) -> Embedding: {embedding_vector.tolist()}")

Input Text: 'anugrah is learning'
Input IDs: tensor([ 1, 38,  5,  9,  1,  0,  0,  0,  0,  0])

Encodings for each token in 'anugrah is learning':
Token: '<unk>' (ID: 1) -> Embedding: [0.05682157352566719, 0.7025406360626221, 0.5732585191726685, 0.7990625500679016, 0.1910640150308609, 0.01756143383681774, 0.4674397110939026, 0.7977325320243835]
Token: 'anugrah' (ID: 38) -> Embedding: [0.6001290082931519, -1.5989844799041748, 0.791266143321991, -0.7446160316467285, 0.2965175211429596, -1.0727243423461914, -0.351509690284729, -0.07790128886699677]
Token: 'is' (ID: 5) -> Embedding: [1.2538002729415894, -1.7399399280548096, -0.5808420181274414, 0.11758750677108765, 2.229707717895508, 0.990931510925293, 0.30065014958381653, -1.5319719314575195]
Token: 'learning' (ID: 9) -> Embedding: [0.5761652588844299, 1.5889612436294556, -0.582440197467804, 0.6181796789169312, -0.4548532962799072, 1.9880694150924683, -0.4141421318054199, 0.1067720279097557]
Token: '<unk>' (ID: 1) -> Embedding: [0.05682157

In [9]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len=5000):
        super().__init__()

        self.dropout = nn.Dropout(0.1)

        # Create a positional encoding matrix
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term) if d_model % 2 == 0 else torch.cos(position * div_term[:-1])

        pe = pe.unsqueeze(0) # Add batch dimension
        self.register_buffer('pe', pe)

    def forward(self, x):
        # Add positional encoding to the input embeddings
        # x has shape (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

# Instantiate Positional Encoding
pos_encoder = PositionalEncoding(d_model, max_seq_len)

# Apply positional encoding to the embedded input
embedded_with_pos = pos_encoder(embedded_input)

print(f"Shape of embedded input with positional encoding: {embedded_with_pos.shape}")

Shape of embedded input with positional encoding: torch.Size([1, 10, 8])


In [10]:
print(embedded_with_pos[0][1],embedded_with_pos.shape) #PE for anugrah (1,batch,10 total , 8 dimension)

tensor([ 1.6018, -1.1763,  0.9901,  0.2782,  0.3406, -0.0809, -0.3895,  1.0246],
       grad_fn=<SelectBackward0>) torch.Size([1, 10, 8])


In [11]:
print(f"Input Text: '{input_text}'")
print(f"Input IDs: {input_ids.squeeze(0)}")

print("\nCombined Embeddings (Word Embedding + Positional Encoding) for each token:")
for i, token_id in enumerate(input_ids.squeeze(0)):
    word = idx_to_word.get(token_id.item(), '<unknown>')
    combined_embedding_vector = embedded_with_pos.squeeze(0)[i]
    print(f"Token: '{word}' (ID: {token_id.item()}) -> Combined Embedding: {combined_embedding_vector.tolist()}")

Input Text: 'anugrah is learning'
Input IDs: tensor([ 1, 38,  5,  9,  1,  0,  0,  0,  0,  0])

Combined Embeddings (Word Embedding + Positional Encoding) for each token:
Token: '<unk>' (ID: 1) -> Combined Embedding: [0.06313508749008179, 1.891711950302124, 0.6369539499282837, 1.9989584684371948, 0.2122933566570282, 1.1306238174438477, 0.5193774700164795, 1.9974807500839233]
Token: 'anugrah' (ID: 38) -> Combined Embedding: [1.6017777919769287, -1.1763136386871338, 0.9901106953620911, 0.27820906043052673, 0.34057486057281494, -0.08086039125919342, -0.38945525884628296, 1.024553656578064]
Token: 'is' (ID: 5) -> Combined Embedding: [2.403441905975342, -2.3956520557403564, -0.4246363639831543, 1.2196156978607178, 2.499673843383789, 2.2119240760803223, 0.33627796173095703, -0.5910821557044983]
Token: 'learning' (ID: 9) -> Combined Embedding: [0.7969836592674255, 0.6655208468437195, -0.31880003213882446, 0.0, -0.47206422686576843, 3.319577217102051, -0.4568246304988861, 1.2297418117523193]
To

In [12]:
token_type_embedding = nn.Embedding(2, d_model) # Assuming 2 types: 0 for sentence A, 1 for sentence B

'''
Segment Embedding

Tells the model which sentence a token belongs to.

Example:

Sentence A: I love AI
Sentence B: It is powerful

BERT input:

[CLS] I love AI [SEP] It is powerful [SEP]
'''

# Get segment embeddings for our token_type_ids
segment_embeddings = token_type_embedding(token_type_ids_new)

# Add the segment embeddings to the combined word and positional embeddings
final_transformer_input = embedded_with_pos + segment_embeddings

print(f"Shape of token_type_ids: {token_type_ids_new.shape}")
print(f"Shape of segment embeddings: {segment_embeddings.shape}")
print(f"Shape of final input to Transformer: {final_transformer_input.shape}")

print("\nSegment Embeddings:")
print(segment_embeddings)

Shape of token_type_ids: torch.Size([1, 10])
Shape of segment embeddings: torch.Size([1, 10, 8])
Shape of final input to Transformer: torch.Size([1, 10, 8])

Segment Embeddings:
tensor([[[-0.1064, -1.4210, -0.3651,  0.5628, -0.5521, -1.9224, -0.5137,
          -0.6295],
         [-0.1064, -1.4210, -0.3651,  0.5628, -0.5521, -1.9224, -0.5137,
          -0.6295],
         [-0.1064, -1.4210, -0.3651,  0.5628, -0.5521, -1.9224, -0.5137,
          -0.6295],
         [-0.1064, -1.4210, -0.3651,  0.5628, -0.5521, -1.9224, -0.5137,
          -0.6295],
         [-0.1064, -1.4210, -0.3651,  0.5628, -0.5521, -1.9224, -0.5137,
          -0.6295],
         [-0.1064, -1.4210, -0.3651,  0.5628, -0.5521, -1.9224, -0.5137,
          -0.6295],
         [-0.1064, -1.4210, -0.3651,  0.5628, -0.5521, -1.9224, -0.5137,
          -0.6295],
         [-0.1064, -1.4210, -0.3651,  0.5628, -0.5521, -1.9224, -0.5137,
          -0.6295],
         [-0.1064, -1.4210, -0.3651,  0.5628, -0.5521, -1.9224, -0.5137,
     

In [13]:
print('final transformer input')
print(final_transformer_input)

final transformer input
tensor([[[-0.0432,  0.4707,  0.2718,  2.5617, -0.3398, -0.7918,  0.0057,
           1.3680],
         [ 1.4954, -2.5973,  0.6250,  0.8410, -0.2115, -2.0033, -0.9032,
           0.3951],
         [ 2.2971, -3.8167, -0.7897,  1.7824,  1.9476,  0.2895, -0.1774,
          -1.2206],
         [ 0.6906, -0.7555, -0.6839,  0.5628, -1.0241,  1.3972, -0.9705,
           0.6002],
         [-0.8841, -1.3667,  0.7045,  2.4740, -0.2953, -1.9224,  0.0101,
          -0.6295],
         [-0.8622, -1.4210, -0.4146,  2.0239,  0.1059, -1.8705, -1.0083,
           0.4035],
         [-0.1072,  0.6165, -0.3199,  1.9659,  0.1170, -1.9224, -1.0072,
           0.4035],
         [ 0.9333,  0.3873, -0.2315,  1.8987,  0.1281, -1.8718, -1.0060,
           0.4035],
         [ 1.3026, -0.6121, -0.1502,  1.8230,  0.1391, -1.8726, -1.0049,
           0.4035],
         [ 0.6612, -1.4628, -0.0769,  1.7395,  0.1502, -1.8736, -1.0038,
          -0.6295]]], grad_fn=<AddBackward0>)


In [14]:
import torch
import torch.nn as nn
import math

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ffn_hidden_dim, dropout_rate=0.1):
        super().__init__()

        '''
          d_model = 512
          num_heads = 8

          head_dim = 64
          512 dimensions
          ├─ Head 1 : 64
          ├─ Head 2 : 64
          ├─ Head 3 : 64
          ├─ Head 4 : 64
          ├─ Head 5 : 64
          ├─ Head 6 : 64
          ├─ Head 7 : 64
          └─ Head 8 : 64
        '''

        assert d_model % num_heads == 0 #if it is divisible then return true and allow run

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Multi-Head Attention
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

        self.attn_dropout = nn.Dropout(dropout_rate)

        # LayerNorm after MHA residual
        self.layer_norm1 = nn.LayerNorm(d_model)

        # Feed Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_hidden_dim),
            nn.ReLU(),
            nn.Linear(ffn_hidden_dim, d_model)
        )

        self.ffn_dropout = nn.Dropout(dropout_rate)

        # LayerNorm after FFN residual
        self.layer_norm2 = nn.LayerNorm(d_model)

    def _calculate_attention(self, Q, K, V, mask=None):

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        ) / math.sqrt(self.head_dim) # The division by sqrt(head_dim) is the scaling factor.

        if mask is not None:
            # Apply padding mask to the scores. Positions where mask == 0 are set to -inf.
            # This prevents attention to padded tokens in the input sequence.
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        output = torch.matmul(
            attention_weights,
            V
        )

        return output

    def forward(self, x, mask=None):
        # In your current setup (cell cbe1d5c1), `mask` is indeed passed as a tensor
        # (attention_mask_new), not None, when calling this layer.

        batch_size, seq_len, _ = x.shape

        # ----------------------------
        # Multi-Head Self Attention
        # ----------------------------

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        '''
        Full token embedding (8 dims)
                  │
                  ▼
            Split into 2 heads
                  │
          ┌──────┴──────┐
          ▼             ▼
        Head 1        Head 2
        (4 dims)      (4 dims)
          │             │
        Attention    Attention
        separately   separately
        '''

        '''
        # Split Q into multiple attention heads and rearrange dimensions
        # From: [batch_size, seq_len, d_model]
        # To:   [batch_size, num_heads, seq_len, head_dim]
        '''
        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        attention_output = self._calculate_attention(
            Q,
            K,
            V,
            mask=mask
        )
        '''
        Before attention:

        Token
        [1,2,3,4,5,6,7,8]

        ↓ split

        Head1 [1,2,3,4]
        Head2 [5,6,7,8]

        After attention:
        Head1 [1,2,3,4]
        Head2 [5,6,7,8]

        ↓ concatenate
        combined attention heads : [1,2,3,4,5,6,7,8]
        '''
        attention_output = (
            attention_output
            .transpose(1, 2)
            .contiguous()
            .view(
                batch_size,
                seq_len,
                self.d_model
            )
        )
        '''
        Multi-head outputs
              ↓
        Concatenate
              ↓
        Wo (Linear Layer)
              ↓
        Final attention representation

        [1,2,3,4,5,6,7,8]
            ↓
        Linear Wo
            ↓
       [5,-2,8,1,4,9,3,7]
        '''

        mha_output = self.Wo(
            attention_output
        )
        '''
        Some output values are randomly set to 0.Prevent Overfitting

        Before:
        [5,-2,8,1,4,9,3,7]

        After dropout:
        [5,0,8,1,0,9,3,7]
        '''
        final_mha_output = self.attn_dropout(
            mha_output
        )

        '''
          Input
            ↓
          Wq, Wk, Wv
            ↓
          Split into heads
            ↓
          Attention
            ↓
          Merge heads
            ↓
          Wo
            ↓
          Dropout
            ↓
          Final MHA output

        '''

        # Residual Connection + LayerNorm
        x = self.layer_norm1(
            x + final_mha_output
        )

        # ----------------------------
        # Feed Forward Network
        # ----------------------------

        ffn_output = self.ffn(x)

        ffn_output = self.ffn_dropout(
            ffn_output
        )

        # Residual Connection + LayerNorm
        output = self.layer_norm2(
            x + ffn_output
        )

        return output

# --- Execution Code ---
encoder_layer = TransformerEncoderLayer(d_model=8, num_heads=2, ffn_hidden_dim=32)
encoder_output = encoder_layer(final_transformer_input, mask=attention_mask_new)

print(f"Encoder Layer Input Shape: {final_transformer_input.shape}")
print(f"Encoder Layer Output Shape: {encoder_output.shape}")
print("\nFirst few values of the output:")
print(encoder_output[0, 0, :])

Encoder Layer Input Shape: torch.Size([1, 10, 8])
Encoder Layer Output Shape: torch.Size([1, 10, 8])

First few values of the output:
tensor([-0.9358,  0.2226,  0.1538,  1.6975, -0.6236, -1.2225, -0.6549,  1.3628],
       grad_fn=<SelectBackward0>)


### Flow of the Transformer Decoder Layer

The `TransformerDecoderLayer` is composed of three main sub-layers, each performing a crucial function in processing the target sequence and integrating information from the encoder.



#### Step 1: Masked Multi-Head Self-Attention

This is the first attention sub-layer within the decoder. It operates on the target sequence itself, allowing each position to attend to all preceding positions (including itself) in the target sequence. A *look-ahead mask* is applied to prevent positions from attending to subsequent positions, ensuring that the prediction for a given token only depends on known outputs. This sub-layer also includes residual connections and layer normalization.

#### Step 2: Multi-Head Encoder-Decoder Attention

This is the second attention sub-layer in the decoder. It's responsible for allowing the decoder to attend to the output of the encoder. Here, the Query (Q) comes from the output of the *first* (masked self-attention) sub-layer of the decoder, while the Key (K) and Value (V) come from the output of the *encoder*. This mechanism enables the decoder to focus on relevant parts of the source sequence during translation or generation. Padding masks from the encoder output are typically applied here. This sub-layer also includes residual connections and layer normalization.

#### Step 3: Feed-Forward Network

Following the two attention sub-layers, a position-wise fully connected Feed-Forward Network (FFN) is applied to each position independently. This FFN typically consists of two linear transformations with a ReLU activation in between. It helps the model learn more complex non-linear relationships. Like the attention sub-layers, it includes residual connections and layer normalization.

In [19]:
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ffn_hidden_dim, dropout_rate=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        '''
            Target Tokens
                  ↓
            Embedding + Positional Encoding
                  ↓
            Masked Self-Attention
                  ↓
            Add & Norm
                  ↓
            Encoder-Decoder Attention
                  ↓
            Add & Norm
                  ↓
            Feed Forward Network
                  ↓
            Add & Norm
                  ↓
            Output
            '''

        # Masked Multi-Head Attention (Self-Attention for Decoder)
        self.masked_Wq = nn.Linear(d_model, d_model)
        self.masked_Wk = nn.Linear(d_model, d_model)
        self.masked_Wv = nn.Linear(d_model, d_model)
        self.masked_Wo = nn.Linear(d_model, d_model)
        self.masked_attn_dropout = nn.Dropout(dropout_rate)
        self.layer_norm1 = nn.LayerNorm(d_model)

        # Multi-Head Attention (Encoder-Decoder Attention)
        self.enc_dec_Wq = nn.Linear(d_model, d_model) # Query comes from decoder
        self.enc_dec_Wk = nn.Linear(d_model, d_model) # Key from encoder output
        self.enc_dec_Wv = nn.Linear(d_model, d_model) # Value from encoder output
        self.enc_dec_Wo = nn.Linear(d_model, d_model)
        self.enc_dec_attn_dropout = nn.Dropout(dropout_rate)
        self.layer_norm2 = nn.LayerNorm(d_model)

        # Feed Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_hidden_dim),
            nn.ReLU(),
            nn.Linear(ffn_hidden_dim, d_model)
        )
        self.ffn_dropout = nn.Dropout(dropout_rate)
        self.layer_norm3 = nn.LayerNorm(d_model)

    def _calculate_attention(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            # Apply mask to the scores. Masked positions (where mask == 0) are set to -inf
            # For decoder self-attention (tgt_mask), this handles both padding and look-ahead (causal) masking.
            # For encoder-decoder attention (src_mask), this handles padding masking from the encoder output.
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = torch.softmax(scores, dim=-1)
        output = torch.matmul(attention_weights, V)
        return output

    def forward(self, x, enc_output, src_mask, tgt_mask):
        batch_size, seq_len, _ = x.shape

        # 1. Masked Multi-Head Self-Attention
        # The tgt_mask here incorporates both padding and look-ahead (causal) masking
        # to ensure that predictions at a given position can only depend on known outputs at earlier positions.
        Q_masked = self.masked_Wq(x)
        K_masked = self.masked_Wk(x)
        V_masked = self.masked_Wv(x)

        Q_masked = Q_masked.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K_masked = K_masked.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V_masked = V_masked.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        masked_attn_output = self._calculate_attention(Q_masked, K_masked, V_masked, mask=tgt_mask)
        masked_attn_output = (masked_attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model))
        mha_output = self.masked_Wo(masked_attn_output)
        mha_output = self.masked_attn_dropout(mha_output)
        x = self.layer_norm1(x + mha_output)

        # 2. Multi-Head Encoder-Decoder Attention
        # The src_mask here is typically just a padding mask from the encoder's input
        # to ensure the decoder doesn't attend to padding tokens in the encoder's output.
        Q_enc_dec = self.enc_dec_Wq(x) # Query from decoder's output of previous sub-layer
        K_enc_dec = self.enc_dec_Wk(enc_output) # Key from encoder output
        V_enc_dec = self.enc_dec_Wv(enc_output) # Value from encoder output

        # Encoder output might have a different sequence length than decoder input 'x'
        # Get encoder sequence length
        enc_seq_len = enc_output.shape[1]

        Q_enc_dec = Q_enc_dec.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K_enc_dec = K_enc_dec.view(batch_size, enc_seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V_enc_dec = V_enc_dec.view(batch_size, enc_seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        enc_dec_attn_output = self._calculate_attention(Q_enc_dec, K_enc_dec, V_enc_dec, mask=src_mask)
        enc_dec_attn_output = (enc_dec_attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model))
        enc_dec_mha_output = self.enc_dec_Wo(enc_dec_attn_output)
        enc_dec_mha_output = self.enc_dec_attn_dropout(enc_dec_mha_output)
        x = self.layer_norm2(x + enc_dec_mha_output)

        # 3. Feed Forward Network
        ffn_output = self.ffn(x)
        ffn_output = self.ffn_dropout(ffn_output)
        output = self.layer_norm3(x + ffn_output)

        return output


class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, ffn_hidden_dim, num_layers, max_seq_len, dropout_rate=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len) # Re-use PositionalEncoding class

        self.decoder_layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, ffn_hidden_dim, dropout_rate)
            for _ in range(num_layers)
        ])

    def forward(self, tgt, enc_output, src_mask, tgt_mask):
        # Embeddings + Positional Encoding
        x = self.embedding(tgt)
        x = self.positional_encoding(x)

        # Pass through decoder layers
        for layer in self.decoder_layers:
            x = layer(x, enc_output, src_mask, tgt_mask)

        return x

In [18]:
import torch
import torch.nn as nn
import math

# Re-defining TransformerEncoder for robustness
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, ffn_hidden_dim, num_layers, max_seq_len, dropout_rate=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len)

        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, ffn_hidden_dim, dropout_rate)
            for _ in range(num_layers)
        ])

    def forward(self, src, src_mask):
        # Embeddings + Positional Encoding
        x = self.embedding(src)
        x = self.positional_encoding(x)

        # Pass through encoder layers
        for layer in self.encoder_layers:
            x = layer(x, mask=src_mask)

        return x

# Re-defining FullTransformer for robustness
class FullTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, ffn_hidden_dim, num_encoder_layers, num_decoder_layers, max_seq_len, dropout_rate=0.1):
        super().__init__()

        self.encoder = TransformerEncoder(
            src_vocab_size, d_model, num_heads, ffn_hidden_dim, num_encoder_layers, max_seq_len, dropout_rate
        )

        self.decoder = TransformerDecoder(
            tgt_vocab_size, d_model, num_heads, ffn_hidden_dim, num_decoder_layers, max_seq_len, dropout_rate
        )

        self.output_linear = nn.Linear(d_model, tgt_vocab_size) # Output layer for predicting target vocabulary

    def forward(self, src, tgt, src_mask, tgt_mask):
        # Encoder forward pass
        enc_output = self.encoder(src, src_mask)

        # Decoder forward pass
        dec_output = self.decoder(tgt, enc_output, src_mask, tgt_mask)

        # Final linear layer to project to target vocabulary size
        output = self.output_linear(dec_output)

        return output

# Model Parameters (re-using existing values from the notebook where appropriate)
# vocab_size, d_model, num_heads, ffn_hidden_dim, num_layers, max_seq_len are already in kernel state.

# Assuming source and target vocabularies are the same for this general example
src_vocab_size_full_transformer = vocab_size
tgt_vocab_size_full_transformer = vocab_size

# Define num_layers here
num_layers = 2 # Example: 2 layers for both encoder and decoder
num_heads = 2 # Example: 2 attention heads
ffn_hidden_dim = 32 # Example: Hidden dimension for Feed Forward Network

# Using num_layers for both encoder and decoder as a starting point
num_encoder_layers_full_transformer = num_layers
num_decoder_layers_full_transformer = num_layers

dropout_rate_full_transformer = 0.1 # Standard dropout rate

# Instantiate the FullTransformer model
full_transformer_model = FullTransformer(
    src_vocab_size=src_vocab_size_full_transformer,
    tgt_vocab_size=tgt_vocab_size_full_transformer,
    d_model=d_model,
    num_heads=num_heads,
    ffn_hidden_dim=ffn_hidden_dim,
    num_encoder_layers=num_encoder_layers_full_transformer,
    num_decoder_layers=num_decoder_layers_full_transformer,
    max_seq_len=max_seq_len,
    dropout_rate=dropout_rate_full_transformer
)

print("FullTransformer instance created successfully!")
print(full_transformer_model)

FullTransformer instance created successfully!
FullTransformer(
  (encoder): TransformerEncoder(
    (embedding): Embedding(39, 8)
    (positional_encoding): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder_layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (Wq): Linear(in_features=8, out_features=8, bias=True)
        (Wk): Linear(in_features=8, out_features=8, bias=True)
        (Wv): Linear(in_features=8, out_features=8, bias=True)
        (Wo): Linear(in_features=8, out_features=8, bias=True)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (layer_norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
        (ffn): Sequential(
          (0): Linear(in_features=8, out_features=32, bias=True)
          (1): ReLU()
          (2): Linear(in_features=32, out_features=8, bias=True)
        )
        (ffn_dropout): Dropout(p=0.1, inplace=False)
        (layer_norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=T